아마존 닷컴 사이트에서 베스트 셀러 상품의 정보를 추출 

아마존 사이트에 접속한다. 
카테고리 분야에서 원하는 카테고리를 선택한다. <- 아됐어 선택하지 말라해 

사용자에게 카테고리 목록을 보여주고
그 목록 들 중에 몇번째 항목을 조회할 것인지 
입력값을 받고
받은 입력값의 항목을 조회하는 코드 


일단 나는 Amazon Renewed 이걸 본다고 치고. 
저 카테고리를 클릭한 뒤, 
페이지 이동을 하면, Amazon Renewed의 베스트셀러가 나온다. 


몇번째 페이지인지 보여주기  
몇번째 내용인지 

판매 순위
판매 순위는 총 100순위 까지 볼 수 있으며, 
스크롤을 내려야지 다음 순위를 더 볼 수 있음. 
스크롤을 내려야 하니까.. 동적으로? selenium을 해야 하는건가?? 
전부 다 볼거니까, 일단 스크롤을 다 내려야 함 
그리고 나서 수집을 함 
순위가 #1 이런 식으로 나와있는데, 이걸 뽑는게 나은가 아니면
위에서 순서대로 1위부터 100위까지이니까 내가 그냥 순위를 메기면 되는건가? 

1페이지 안에, 
1순위부터 50순위가 포함되어있는 상품 전체를 잡고, 
첫번째 상품 안에, 순위, 제품이름, 가격, 상품평 수, 평점 뽑고, 
다음 상품도 마찬가지로 뽑고 뽑고,
그러다가 50개가 넘어가면 다음 페이지에 가서 이어서 뽑고  


A. 상품 카드 1개 찾기 ~ 상품 카드 하나 → 그 안에서 순위 추출”
먼저 상품 카드 하나를 제대로 잡는지 확인
B. 상품 카드 여러 개 찾기
현재 몇 개까지 잡히는지 확인
C. 스크롤하면서 100개까지 늘리기
상품 카드 개수가 증가하는지 확인
D. 첫 번째 상품에서 항목 하나씩 테스트
순위 
제품명
가격
리뷰 수
평점
E. 전체 반복문으로 확장




ol (상품 리스트)
 ├─ li (상품 카드)
 │   └─ span.zg-bdg-text   ← 순위 (#1)
 │   └─ 상품명
 │   └─ 가격
 │   └─ 평점 
 │   └─ 리뷰수



<div class="a-section zg-bdg-body zg-bdg-clr-body aok-float-left"><span class="zg-bdg-text">#1</span></div>


순위 zg-bdg-text
제품이름 _cDEzb_p13n-sc-css-line-clamp-3_g3dy1
가격 _cDEzb_p13n-sc-price_3mJ9Z
상품평 수  a-size-small
평점  a-icon-alt


한 페이지에 총 50개까지 있고
그 다음 내용을 뽑으려면 
다음 페이지를 클릭한다. 
이 과정은 반복문으로 진행해야 겠지?

그리고 다시 목록으로 돌아가야 하나? 또 다른 카테고리를 고려면 ?
이어서 카테고리를 더 보시겠습니까?
아 아니면 보고 싶은 카테고리 번호를 다 고르라고 해야 하나 ? 


다 하고 나면, csv , xlsx 형태로 저장한다. 

In [ ]:
# Step 1.필요한 라이브러리 로딩
import urllib.request , urllib
import pandas as pd    
from bs4 import BeautifulSoup
import os, tempfile, time, math
import undetected_chromedriver as uc   # pip install undetected-chromedriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException


In [ ]:
#Step 2. 사용자에게 검색어 키워드를 입력 받습니다.
print("=" *80)
print(" 아마존 사이트의 카테고리별 Best Seller 상품 정보 추출하기 ")
print("=" *80)

cnt = int(input('1.크롤링 할 건수는 몇건입니까?: '))
page_cnt = math.ceil(cnt/50)

f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본경로:c:\\py_temp\\):")
if f_dir == '' :
    f_dir = "c:\\py_temp\\"
    
print("\n")

if cnt > 60 :
      print("요청 건수가 많아서 시간이 제법 소요되오니 잠시만 기다려 주세요~~")
else :
      print("요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~")

 아마존 사이트의 카테고리별 Best Seller 상품 정보 추출하기 




요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~


In [ ]:
#Step 3.저장될 파일 경로와 이름을 지정합니다 # 이미지 파일, 텍스트 파일 따로 저장하고 나중에 합침
sec_name = '베스트셀러'
query_txt='아마존'

n = time.localtime()
s1 = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s1+'-'+query_txt+'-'+sec_name)
os.chdir(f_dir+s1+'-'+query_txt+'-'+sec_name)

ff_dir=f_dir+s1+'-'+query_txt+'-'+sec_name
ff_name=f_dir+s1+'-'+query_txt+'-'+sec_name+'\\'+s1+'-'+query_txt+'-'+sec_name+'.txt'
fc_name=f_dir+s1+'-'+query_txt+'-'+sec_name+'\\'+s1+'-'+query_txt+'-'+sec_name+'.csv'
fx_name=f_dir+s1+'-'+query_txt+'-'+sec_name+'\\'+s1+'-'+query_txt+'-'+sec_name+'.xlsx'

# 제품 이미지 저장용 폴더 생성
img_dir = ff_dir+"\\images"
os.makedirs(img_dir)
os.chdir(img_dir)
    
s_time = time.time( )


In [ ]:
# Step 4. 중요!!  undetected chromedriver 설정하기
CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"  # 크롬 경로 확인!
VERSION_MAIN = 145  # 현재 크롬 메이저 버전 ~ 버전이 바뀌면 그때 또 수정하면 됨 

options = Options()
# 필요하면 headless: 많은 사이트가 탐지하니 처음엔 꺼두고 테스트
# options.add_argument("--headless=new")

# 깨끗한 임시 프로필(기존 프로필 충돌 방지)
user_data = os.path.join(tempfile.gettempdir(), "uc_profile_clean")
os.makedirs(user_data, exist_ok=True)

driver = uc.Chrome(
    options=options,
    browser_executable_path=CHROME_PATH,
    version_main=VERSION_MAIN,      # ← 핵심: 140으로 맞추기
    user_data_dir=user_data,
    use_subprocess=False,           # 안 되면 True로 바꿔 재시도
    patcher_force_close=True,       # 패쳐가 점유 중일 때 강제 종료
    debug=True,
)
# 여기까지 그대로 복붙해서 쓰면 됨 
# 이전까지는 카테고리 누르고, 식품 눌렀는데, 이제는 링크로 이동하는구나?? 
driver.get("https://www.amazon.com/bestsellers?ld=NSGoogle")
driver.maximize_window()
time.sleep(3)



In [ ]:
# ul _p13n-zg-nav-tree-all_style_zg-browse-group__88fbz
# li 


# category_name = input("원하는 카테고리를 입력하세요: ").strip().lower()
# keyword = input("검색할 상품명을 입력하세요: ").strip()




In [ ]:
# 카테고리 li 전체 가져오기
# items = driver.find_elements(
#     By.CSS_SELECTOR,
#     'ul._p13n-zg-nav-tree-all_style_zg-browse-root__-jwNv li'
# )
# items = driver.find_elements(
#     By.XPATH, '//*[@id="CardInstancewiS9luo_xgrCJb2zdaoB_Q"]/ul/li[2]/ul'
# )

items = driver.find_elements(
    By.CSS_SELECTOR,
    'ul._p13n-zg-nav-tree-all_style_zg-browse-group__88fbz > li'
)

category_list = []
element_list = []

print("===== 카테고리 목록 =====")

for item in items:

    try:
        a = item.find_element(By.TAG_NAME, "a")
        name = a.text.strip()

        if name != "":
            category_list.append(name)
            element_list.append(a)

            print(len(category_list)-1, ":", name)

    except:
        continue

# 사용자 입력
num = int(input("조회할 카테고리 번호 입력: "))

# 해당 카테고리 클릭
element_list[num].click()

print(category_list[num], "카테고리 선택 완료")

time.sleep(5)

===== 카테고리 목록 =====
0 : Amazon Devices & Accessories
1 : Amazon Renewed
2 : Appliances
3 : Apps & Games
4 : Arts, Crafts & Sewing
5 : Audible Books & Originals
6 : Automotive
7 : Baby
8 : Beauty & Personal Care
9 : Books
10 : Camera & Photo Products
11 : CDs & Vinyl
12 : Cell Phones & Accessories
13 : Clothing, Shoes & Jewelry
14 : Collectible Coins
15 : Computers & Accessories
16 : Digital Educational Resources
17 : Digital Music
18 : Electronics
19 : Entertainment Collectibles
20 : Gift Cards
21 : Grocery & Gourmet Food
22 : Handmade Products
23 : Health & Household
24 : Home & Kitchen
25 : Industrial & Scientific
26 : Kindle Store
27 : Kitchen & Dining
28 : Movies & TV
29 : Musical Instruments
30 : Office Products
31 : Patio, Lawn & Garden
32 : Pet Supplies
33 : Software
34 : Sports & Outdoors
35 : Sports Collectibles
36 : Tools & Home Improvement
37 : Toys & Games
38 : Unique Finds
39 : Video Games
Amazon Devices & Accessories 카테고리 선택 완료


In [ ]:
# 페이지 전체에서 span만 찾기 
# 이 span이 어떤 상품 카드에 속하는지 정보가 약함
ranks = driver.find_elements(By.CSS_SELECTOR, "span.zg-bdg-text")

for r in ranks:
    print(r.text)

#1
#2
#3
#4
#5
#6
#7
#8
#9
#10
#11
#12
#13
#14
#15
#16
#17
#18
#19
#20
#21
#22
#23
#24
#25
#26
#27
#28
#29
#30


In [ ]:
# 상품 카드들은 잡고, 그 다음 각 카드 안에 순위를 찾는다. 
# 순위를 어떤 상품에서 찾았는지가 분명함
items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i in range(len(items)):
    items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')
    rank = items[i].find_element(By.CSS_SELECTOR, "span.zg-bdg-text").text
    print(rank)

#1
#2
#3
#4
#5
#6
#7
#8
#9
#10
#11
#12
#13
#14
#15
#16
#17
#18
#19
#20
#21
#22
#23
#24
#25
#26
#27
#28
#29
#30


In [ ]:
def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,1100);")
    time.sleep(1)

items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i in range(len(items)):
    items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')
    item = items[i]
    rank = item.find_element(By.CSS_SELECTOR, "span.zg-bdg-text").text
    print(rank)

#1
#2
#3
#4
#5
#6
#7
#8
#9
#10
#11
#12
#13
#14
#15
#16
#17
#18
#19
#20
#21
#22
#23
#24
#25
#26
#27
#28
#29
#30


In [ ]:
for item in items:

    rank = item.find_element(By.CSS_SELECTOR, "span.zg-bdg-text").text

    if rank == "#1":
        print("1위 상품 발견")

1위 상품 발견


In [ ]:
def scroll_down(driver):
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
    time.sleep(3)

for i in range(5):
    scroll_down(driver)

# 페이지 전체에서 상품 카드들을 전부 가져옴
# 상품 카드 하나씩 꺼냄
# 현재 상품 카드 내부에서만 상품명을 찾음

# 상품 1개 박스: div[id="gridItemRoot"]
# 그 안의 상품명: div[class*="p13n-sc-css-line-clamp-3"]
items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for item in items:
    title = item.find_element(By.CSS_SELECTOR, 'div[class*="p13n-sc-css-line-clamp-3"]').text
    print(title)

blink plus plan with monthly auto-renewal
Amazon Fire TV Stick 4K Select (newest model), start streaming in 4K, AI-powered search, and free & live TV
Amazon Fire TV Stick 4K Plus (newest model) with AI-powered Fire TV Search, Wi-Fi 6, stream over 1.8 million movies and shows, free & live TV
Ring Battery Doorbell (newest model), Home or business security with Head-to-Toe video, Live View with Two-Way Talk, and Motion Detection & Alerts, Satin Nickel
Amazon Kindle Paperwhite 16GB (newest model) – 20% faster, with new 7" glare-free display and weeks of battery life – Black
Amazon Fire TV Stick 4K Max streaming device, with AI-powered Fire TV Search, supports Wi-Fi 6E, free & live TV without cable or satellite
Amazon Echo Dot (newest model) - Vibrant sounding speaker, Designed for Alexa+, Great for bedrooms, dining rooms and offices, Charcoal
Amazon Fire TV Stick HD (newest model), free and live TV, Alexa Voice Remote, smart home controls, HD streaming
Blink Video Doorbell (newest model) –

In [ ]:
def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,1100);")
    time.sleep(1)


items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for item in items:
    title = item.find_element(By.CSS_SELECTOR,
                              'div._cDEzb_p13n-sc-css-line-clamp-3_g3dy1').text
    print(title)

blink plus plan with monthly auto-renewal
Amazon Fire TV Stick 4K Select (newest model), start streaming in 4K, AI-powered search, and free & live TV
Amazon Fire TV Stick 4K Plus (newest model) with AI-powered Fire TV Search, Wi-Fi 6, stream over 1.8 million movies and shows, free & live TV
Ring Battery Doorbell (newest model), Home or business security with Head-to-Toe video, Live View with Two-Way Talk, and Motion Detection & Alerts, Satin Nickel
Amazon Kindle Paperwhite 16GB (newest model) – 20% faster, with new 7" glare-free display and weeks of battery life – Black
Amazon Fire TV Stick 4K Max streaming device, with AI-powered Fire TV Search, supports Wi-Fi 6E, free & live TV without cable or satellite
Amazon Echo Dot (newest model) - Vibrant sounding speaker, Designed for Alexa+, Great for bedrooms, dining rooms and offices, Charcoal
Amazon Fire TV Stick HD (newest model), free and live TV, Alexa Voice Remote, smart home controls, HD streaming
Blink Video Doorbell (newest model) –

In [ ]:
from selenium.webdriver.common.by import By
import time

# 먼저 페이지 맨 아래까지 몇 번 내려보기
for i in range(3):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i, item in enumerate(items, start=1):
    try:
        title = item.find_element(By.CSS_SELECTOR, 'div[class*="p13n-sc-css-line-clamp-3"]').text
        print(i, title)
    except:
        print(i, "상품명 없음")

1 blink plus plan with monthly auto-renewal
2 Amazon Fire TV Stick 4K Select (newest model), start streaming in 4K, AI-powered search, and free & live TV
3 Amazon Fire TV Stick 4K Plus (newest model) with AI-powered Fire TV Search, Wi-Fi 6, stream over 1.8 million movies and shows, free & live TV
4 Ring Battery Doorbell (newest model), Home or business security with Head-to-Toe video, Live View with Two-Way Talk, and Motion Detection & Alerts, Satin Nickel
5 Amazon Kindle Paperwhite 16GB (newest model) – 20% faster, with new 7" glare-free display and weeks of battery life – Black
6 Amazon Fire TV Stick 4K Max streaming device, with AI-powered Fire TV Search, supports Wi-Fi 6E, free & live TV without cable or satellite
7 Amazon Echo Dot (newest model) - Vibrant sounding speaker, Designed for Alexa+, Great for bedrooms, dining rooms and offices, Charcoal
8 Amazon Fire TV Stick HD (newest model), free and live TV, Alexa Voice Remote, smart home controls, HD streaming
9 Blink Video Doorbel

In [ ]:
from selenium.webdriver.common.by import By
import time

# 먼저 페이지 맨 아래까지 몇 번 내려보기
for i in range(3):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i, item in enumerate(items, start=1):
    try:
        title = item.find_element(By.CSS_SELECTOR, 'span[class*="_cDEzb_p13n-sc-price_3mJ9Z"]').text
        print(i, title)
    except:
        print(i, "가격 없음")

1 $11.99
2 가격 없음
3 가격 없음
4 가격 없음
5 가격 없음
6 가격 없음
7 가격 없음
8 가격 없음
9 가격 없음
10 가격 없음
11 가격 없음
12 가격 없음
13 가격 없음
14 가격 없음
15 가격 없음
16 가격 없음
17 가격 없음
18 가격 없음
19 가격 없음
20 $149.99
21 가격 없음
22 가격 없음
23 가격 없음
24 가격 없음
25 가격 없음
26 가격 없음
27 가격 없음
28 가격 없음
29 가격 없음
30 가격 없음
31 가격 없음
32 $5.99
33 가격 없음
34 가격 없음
35 가격 없음
36 가격 없음
37 가격 없음
38 가격 없음
39 가격 없음
40 가격 없음
41 가격 없음
42 가격 없음
43 가격 없음
44 가격 없음
45 가격 없음
46 가격 없음
47 가격 없음
48 $9.99
49 가격 없음
50 가격 없음


In [ ]:
items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')
print("item 개수:", len(items))

prices = driver.find_elements(By.CSS_SELECTOR, 'span[class*="p13n-sc-price"]')
print("price 개수:", len(prices))

item 개수: 50
price 개수: 50


In [ ]:
prices = driver.find_elements(By.CSS_SELECTOR, 'span[class*="p13n-sc-price"]')

print("가격 개수:", len(prices))

for i, p in enumerate(prices, start=1):
    print(i, p.text)

가격 개수: 50
1 $11.99
2 $39.99
3 $49.99
4 $59.99
5 $159.99
6 $59.99
7 $49.99
8 $34.99
9 $59.99
10 $189.99
11 $70.33
12 $49.99
13 $89.99
14 $44.99
15 $199.99
16 $33.24
17 $79.99
18 $34.99
19 $139.99
20 $149.99
21 $199.99
22 $99.99
23 $49.99
24 $219.99
25 $17.56
26 $67.99
27 $24.99
28 $149.99
29 $109.99
30 $57.99
31 $99.99
32 $5.99
33 $179.99
34 $24.99
35 $59.99
36 $189.99
37 $39.99
38 $79.99
39 $59.99
40 $59.99
41 $17.56
42 $59.99
43 $34.99
44 $44.99
45 $26.59
46 $79.99
47 $95.62
48 $9.99
49 $224.99
50 $49.99


In [ ]:
# 가격 
# <span class="a-size-base a-color-price"><span class="_cDEzb_p13n-sc-price_3mJ9Z">$11.99</span></span>

price = item.find_element(
    By.CSS_SELECTOR,
    'span[class*="p13n-sc-price"]'
).text

price

'$49.99'

In [ ]:
# 상품평 수 
# a-size-small

# 페이지 전체의 a-size-small을 다 가져오게 됩니다.
# driver.find_elements(By.CSS_SELECTOR, ".a-size-small")

prices = driver.find_elements(By.CSS_SELECTOR, 'span[class*="a-size-small"]')

print("상품평 수:", len(prices))

for i, p in enumerate(prices, start=1):
    print(i, p.text)

상품평 수: 52
1 272,677
2 5,594
3 104,917
4 44,429
5 17,525
6 74,272
7 in Amazon Devices & Accessories
8 in Amazon Devices & Accessories
9 185,407
10 60,672
11 19,802
12 24,006
13 14,971
14 39,781
15 65,382
16 1,938
17 7,352
18 126,607
19 40,038
20 1,271
21 40,659
22 169
23 9,997
24 6,888
25 79,520
26 2,928
27 11,667
28 61,539
29 308,433
30 22,906
31 32,135
32 1,395
33 2,133
34 1,670
35 42,579
36 571,769
37 894
38 10,259
39 104,235
40 2,790
41 57,118
42 12,271
43 94,791
44 6,921
45 2,201
46 12,050
47 22,659
48 28,297
49 15,852
50 164
51 4,737
52 49,517


In [ ]:
items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i, item in enumerate(items, start=1):
    try:
        review_count = item.find_element(
            By.CSS_SELECTOR,
            'div.a-icon-row a.a-link-normal span.a-size-small'
        ).text.strip()
        print(i, review_count)
    except:
        print(i, "리뷰 수 없음")

1 272,677
2 5,594
3 104,917
4 44,429
5 17,525
6 74,272
7 185,407
8 60,672
9 19,802
10 24,006
11 14,971
12 39,781
13 65,382
14 1,938
15 7,352
16 126,607
17 40,038
18 1,271
19 40,659
20 169
21 9,997
22 6,888
23 79,520
24 2,928
25 11,667
26 61,539
27 308,433
28 22,906
29 32,135
30 1,395
31 2,133
32 1,670
33 42,579
34 571,769
35 894
36 10,259
37 104,235
38 2,790
39 57,118
40 12,271
41 94,791
42 6,921
43 2,201
44 12,050
45 22,659
46 28,297
47 15,852
48 164
49 4,737
50 49,517


In [ ]:
# 평점
# a-icon-alt
 
prices = driver.find_elements(By.CSS_SELECTOR, 'span[class*="a-icon-alt"]')

print("평점:", prices)

for i, p in enumerate(prices, start=1):
    print(i, p.text)

평점: [<undetected_chromedriver.webelement.WebElement (session="c22d7ba712093820e9996cd975b30672", element="f.2CE9E6DDB14732B7201978A2D2698A8B.d.D36180C48B7737F25A9C49FE08FC08E5.e.625")>, <undetected_chromedriver.webelement.WebElement (session="c22d7ba712093820e9996cd975b30672", element="f.2CE9E6DDB14732B7201978A2D2698A8B.d.D36180C48B7737F25A9C49FE08FC08E5.e.626")>, <undetected_chromedriver.webelement.WebElement (session="c22d7ba712093820e9996cd975b30672", element="f.2CE9E6DDB14732B7201978A2D2698A8B.d.D36180C48B7737F25A9C49FE08FC08E5.e.627")>, <undetected_chromedriver.webelement.WebElement (session="c22d7ba712093820e9996cd975b30672", element="f.2CE9E6DDB14732B7201978A2D2698A8B.d.D36180C48B7737F25A9C49FE08FC08E5.e.628")>, <undetected_chromedriver.webelement.WebElement (session="c22d7ba712093820e9996cd975b30672", element="f.2CE9E6DDB14732B7201978A2D2698A8B.d.D36180C48B7737F25A9C49FE08FC08E5.e.629")>, <undetected_chromedriver.webelement.WebElement (session="c22d7ba712093820e9996cd975b30672"

In [ ]:
items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i, item in enumerate(items, start=1):
    try:
        rating = item.find_element(By.CSS_SELECTOR, "span.a-icon-alt").text
        rating = rating.split(" ")[0]
        print(i, rating)
    except:
        print(i, "평점 없음")

1 
2 
3 
4 
5 
6 
7 
8 
9 
10 
11 
12 
13 
14 
15 
16 
17 
18 
19 
20 
21 
22 
23 
24 
25 
26 
27 
28 
29 
30 
31 
32 
33 
34 
35 
36 
37 
38 
39 
40 
41 
42 
43 
44 
45 
46 
47 
48 
49 
50 


In [ ]:
rating = item.find_element(By.CSS_SELECTOR, "span.a-icon-alt")
print(rating.text)
print(rating.get_attribute("innerHTML"))


4.4 out of 5 stars


In [ ]:
items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i, item in enumerate(items, start=1):
    try:
        rating = item.find_element(
            By.CSS_SELECTOR,
            'a[aria-label*="stars"]'
        ).get_attribute("aria-label")

        rating = rating.split(" ")[0]

        print(i, rating)
    except:
        print(i, "평점 없음")

1 4.4
2 4.0
3 4.7
4 4.6
5 4.7
6 4.6
7 4.7
8 4.7
9 4.2
10 4.2
11 4.6
12 4.7
13 4.2
14 4.4
15 4.3
16 4.7
17 4.6
18 4.6
19 4.5
20 4.6
21 4.6
22 4.5
23 4.4
24 4.2
25 4.7
26 4.6
27 4.4
28 4.5
29 4.4
30 3.9
31 4.3
32 4.5
33 4.7
34 4.7
35 4.5
36 4.6
37 4.7
38 4.1
39 4.6
40 4.6
41 4.6
42 4.7
43 4.7
44 4.2
45 4.5
46 4.4
47 4.5
48 4.2
49 4.4
50 4.4


1단계. 한 페이지에서 상품 카드 50개가 잡히는지 먼저 확인

먼저 이걸 봐야 합니다.

현재 페이지에서 상품 카드가 정말 50개 잡히는지

중간에 빠지는 카드가 없는지

광고 카드, 추천 카드, 빈 카드가 섞이는지



2단계. 그 50개 각각에서 필요한 항목 4~5개를 다 뽑아보기

이미 상품 하나에서 해봤다고 했으니까,
이제는 전체 50개를 반복문으로 돌렸을 때도 안정적인지 확인해야 합니다.

즉, “한 개는 된다”와 “50개 전부 된다”는 완전히 다른 문제예요.

왜냐하면 실제로는 이런 일이 많이 생깁니다.

어떤 상품은 가격이 없음

어떤 상품은 리뷰 수가 없음

어떤 상품은 평점이 없음

어떤 상품은 구조가 조금 다름

어떤 상품은 sponsored 카드일 수 있음

그래서 지금은 이런 걸 확인해야 합니다.

1번 상품: 순위/제품명/가격/리뷰수/평점 잘 나오는가

2번 상품: 잘 나오는가

...

50번 상품까지 잘 나오는가

즉, 전체 반복문 테스트가 필요합니다.


Step 3. 누락되는 항목 selector 수정

Step 4. 한 페이지 결과를 리스트에 저장

Step 5. 그다음에야 다음 페이지 클릭 테스트

In [ ]:
def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,1100);")
    time.sleep(1)

# def scroll_down(driver):
#     driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
#     time.sleep(5)

# 그냥 단순하게 내려보기만 하는 거 
# for _ in range(5):
#     driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
#     time.sleep(2)

items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')
print("상품 카드 수:", len(items))


상품 카드 수: 50


In [ ]:
items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')

for i, item in enumerate(items, start=1):
    try:
        rank = item.find_element(By.CSS_SELECTOR, "span.zg-bdg-text").text
    except:
        rank = ""

    try:
        title = item.find_element(
            By.CSS_SELECTOR,
            'div[class*="p13n-sc-css-line-clamp-3"]'
        ).text
    except:
        title = ""

    try:
        price = item.find_element(
            By.CSS_SELECTOR,
            'span[class*="p13n-sc-price"]'
        ).text
    except:
        price = ""

    try:
        review_count = item.find_element(
            By.CSS_SELECTOR,
            "div.a-icon-row span.a-size-small"
        ).text
    except:
        review_count = ""

    try:
        rating = item.find_element(
            By.CSS_SELECTOR,
            'a[aria-label*="stars"]'
        ).get_attribute("aria-label")

        rating = rating.split(" ")[0]
    except:
        rating = ""

    print(i, rank, title, price, review_count, rating)

1 #1 blink plus plan with monthly auto-renewal $11.99 272,677 4.4
2 #2 Amazon Fire TV Stick 4K Select (newest model), start streaming in 4K, AI-powered search, and free & live TV $39.99 5,594 4.0
3 #3 Amazon Fire TV Stick 4K Plus (newest model) with AI-powered Fire TV Search, Wi-Fi 6, stream over 1.8 million movies and shows, free & live TV $49.99 104,917 4.7
4 #4 Ring Battery Doorbell (newest model), Home or business security with Head-to-Toe video, Live View with Two-Way Talk, and Motion Detection & Alerts, Satin Nickel $59.99 44,429 4.6
5 #5 Amazon Kindle Paperwhite 16GB (newest model) – 20% faster, with new 7" glare-free display and weeks of battery life – Black $159.99 17,525 4.7
6 #6 Amazon Fire TV Stick 4K Max streaming device, with AI-powered Fire TV Search, supports Wi-Fi 6E, free & live TV without cable or satellite $59.99 74,272 4.6
7 #7 Amazon Echo Dot (newest model) - Vibrant sounding speaker, Designed for Alexa+, Great for bedrooms, dining rooms and offices, Charcoal $49.

현재 페이지에서 스크롤 내려서 상품 카드가 충분히 로드되게 함

현재 페이지 상품들 수집

50개를 다 못 채웠으면 다음 페이지로 이동

다음 페이지에서도 같은 작업 반복

원하는 건수 cnt만큼 모이면 종료

즉 구조는 페이지 반복문 안에 상품 반복문이 들어가는 2중 구조로 가면 돼.

----------------------------------------------------------
all_data = []

for page in range(1, page_cnt + 1):   # 페이지 반복
    스크롤 내리기

    items = 상품 카드들 찾기

    for item in items:                # 상품 반복
        상품 정보 추출
        all_data에 저장

        if len(all_data) == cnt:
            종료

    다음 페이지 클릭

------------------------------------------------
for 페이지:
    스크롤

    for 상품:
        순위, 상품명, 가격, 리뷰수, 평점 추출
        리스트에 저장

    다음 페이지 클릭

즉,

바깥 for문 = 페이지 이동 담당

안쪽 for문 = 현재 페이지 상품 50개 담당

이렇게 나누면 돼.

In [ ]:
# items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')
# print("상품 카드 수:", len(items))

# rank_list = []
# title_list = []
# price_list = []
# review_count_list = []
# rating_list = []

# for i, item in enumerate(items, start=1):
#     try:
#         rank = item.find_element(By.CSS_SELECTOR, "span.zg-bdg-text").text
#     except:
#         rank = ""

#     try:
#         title = item.find_element(
#             By.CSS_SELECTOR,
#             'div[class*="p13n-sc-css-line-clamp-3"]'
#         ).text
#     except:
#         title = ""

#     try:
#         price = item.find_element(
#             By.CSS_SELECTOR,
#             'span[class*="p13n-sc-price"]'
#         ).text
#     except:
#         price = ""

#     try:
#         review_count = item.find_element(
#             By.CSS_SELECTOR,
#             'div.a-icon-row span.a-size-small'
#         ).text
#     except:
#         review_count = ""

#     try:
#         rating = item.find_element(
#             By.CSS_SELECTOR,
#             'a[aria-label*="stars"]'
#         ).get_attribute("aria-label")
#         rating = rating.split(" ")[0]
#     except:
#         rating = ""

#     rank_list.append(rank)
#     title_list.append(title)
#     price_list.append(price)
#     review_count_list.append(review_count)
#     rating_list.append(rating)

# print("rank_list 길이:", len(rank_list))
# print("title_list 길이:", len(title_list))
# print("price_list 길이:", len(price_list))
# print("review_count_list 길이:", len(review_count_list))
# print("rating_list 길이:", len(rating_list))

# co_best_seller = pd.DataFrame({
#     '순위': rank_list,
#     '상품명': title_list,
#     '가격': price_list,
#     '상품평수': review_count_list,
#     '평점': rating_list
# })

# print("데이터프레임 크기:", co_best_seller.shape)
# print(co_best_seller.head())
# print(co_best_seller.tail())



In [ ]:
all_rank = []
all_title = []
all_price = []
all_review_count = []
all_rating = []
all_page = []
all_item_no = []

for page_no in range(1, page_cnt + 1):
    print(f"\n===== {page_no} 페이지 수집 시작 =====")

    # 1. 페이지 아래까지 스크롤
    scroll_down_page(driver)

    # 2. 현재 페이지 상품 카드 찾기
    items = driver.find_elements(By.CSS_SELECTOR, 'div[id="gridItemRoot"]')
    print("현재 페이지 상품 카드 수:", len(items))

    # 3. 현재 페이지 상품 반복
    for i, item in enumerate(items, start=1):
        try:
            rank = item.find_element(By.CSS_SELECTOR, "span.zg-bdg-text").text
        except:
            rank = ""

        try:
            title = item.find_element(
                By.CSS_SELECTOR,
                'div[class*="p13n-sc-css-line-clamp-3"]'
            ).text
        except:
            title = ""

        try:
            price = item.find_element(
                By.CSS_SELECTOR,
                'span[class*="p13n-sc-price"]'
            ).text
        except:
            price = ""

        try:
            review_count = item.find_element(
                By.CSS_SELECTOR,
                'div.a-icon-row span.a-size-small'
            ).text
        except:
            review_count = ""

        try:
            rating = item.find_element(
                By.CSS_SELECTOR,
                'a[aria-label*="stars"]'
            ).get_attribute("aria-label")
            rating = rating.split(" ")[0]
        except:
            rating = ""

        print(page_no, i, rank, title, price, review_count, rating)

        all_page.append(page_no)
        all_item_no.append(i)
        all_rank.append(rank)
        all_title.append(title)
        all_price.append(price)
        all_review_count.append(review_count)
        all_rating.append(rating)

        # 요청 건수만큼 모였으면 중단
        if len(all_rank) >= cnt:
            break

    # 요청 건수 다 채웠으면 바깥 반복문도 종료
    if len(all_rank) >= cnt:
        break

    # 4. 다음 페이지 이동
    moved = go_next_page(driver)
    if not moved:
        print("다음 페이지가 없어서 종료합니다.")
        break

In [ ]:
# csv 형태로 저장하기
co_best_seller.to_csv(fc_name, encoding="utf-8-sig", index=False)

# 엑셀 형태로 저장하기
co_best_seller.to_excel(fx_name, index=False, engine='openpyxl')



e_time = time.time()
t_time = e_time - s_time

print("\n")
print("=" * 100)
print("1. 요청된 총 %s 건 중에서 실제 크롤링 된 건수는 %s 건입니다" % (cnt, len(co_best_seller)))
print("2. 총 소요시간은 %s 초 입니다" % round(t_time, 1))
print("3. csv 파일명 : %s" % fc_name)
print("4. xlsx 파일명 : %s" % fx_name)
print("=" * 100)